# Notebook 4 — Interpretability: Grad-CAM and Failure Analysis

Grad-CAM heatmaps for one image per breed, plus a gallery of the most
confident binary misclassifications.

Because the model is now a 36-way softmax, the "restricted" probability
is the sum of restricted-class softmax outputs. Grad-CAM is computed
against this collapsed scalar so the heatmap shows what pushed the
model toward *restricted* (vs unrestricted) — not just toward one breed.


In [ ]:
# ── Mount Google Drive ─────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')


In [ ]:
# ── Verify GPU ────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detected: {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('No GPU — go to Runtime -> Change runtime type -> T4 GPU')


In [ ]:
# ── Project root on Drive ─────────────────────────────────
from pathlib import Path

ROOT = Path('/content/drive/MyDrive/MSc_Capstone')
for d in ['data', 'pretrained/checkpoints', 'pretrained/logs',
          'pretrained/curves', 'pretrained/inference', 'report']:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

DATA   = ROOT / 'data'
CKPT   = ROOT / 'pretrained' / 'checkpoints'
LOGS   = ROOT / 'pretrained' / 'logs'
CURVES = ROOT / 'pretrained' / 'curves'
INFER  = ROOT / 'pretrained' / 'inference'
REPORT = ROOT / 'report'

print(f'Project root: {ROOT}')
print(f'Data folder:  {DATA}')


In [ ]:
# ── Load breed manifest written by Notebook 1 ────────────────
import json, numpy as np

with open(DATA / 'breed_to_restricted.json') as f:
    MANIFEST = json.load(f)

SELECTED_BREEDS = MANIFEST['breeds']
RESTRICTED_SET  = set(MANIFEST['restricted'])
NUM_CLASSES     = len(SELECTED_BREEDS)
BREED_TO_IDX    = {b: i for i, b in enumerate(SELECTED_BREEDS)}

# Boolean mask aligned with class index order used by image_dataset_from_directory
# (alphabetical). image_dataset_from_directory sorts class names alphabetically,
# so we mirror that ordering here.
CLASS_NAMES     = sorted(SELECTED_BREEDS)
RESTRICTED_MASK = np.array([(b in RESTRICTED_SET) for b in CLASS_NAMES], dtype=bool)
RESTRICTED_IDX  = np.where(RESTRICTED_MASK)[0]

print(f'{NUM_CLASSES} classes loaded.')
print(f'Restricted indices (in alphabetical order): {RESTRICTED_IDX.tolist()}')


## 4.1 — Load the best model


In [ ]:
import os, glob, random, cv2
from collections import defaultdict
import matplotlib.pyplot as plt

model_path = str(CKPT / 'FineTuned_best.keras')
if not os.path.exists(model_path):
    model_path = str(CKPT / 'InceptionResNetV2_best.keras')
if not os.path.exists(model_path):
    raise FileNotFoundError(f'No model found in {CKPT}')

model = tf.keras.models.load_model(model_path)
print(f'Loaded model: {os.path.basename(model_path)}')

base_model = None
for layer in model.layers:
    if isinstance(layer, tf.keras.Model):
        base_model = layer
        break

conv_layers = [l.name for l in base_model.layers if 'conv' in l.name]
LAST_CONV = conv_layers[-1]
print(f'Base: {base_model.name}  | Last conv: {LAST_CONV}')

GRADCAM_DIR = INFER / 'gradcam_images'
GRADCAM_DIR.mkdir(parents=True, exist_ok=True)


## 4.2 — Grad-CAM function (multi-class softmax → restricted scalar)


In [ ]:
# The saved model already contains its preprocess_input layer, so feed
# raw 0–255 images. Split the model into pre/post layers around the
# nested base_model so we can tap the last conv activations.
RESTRICTED_IDX_TF = tf.constant(RESTRICTED_IDX, dtype=tf.int32)
BASE_IDX    = next(i for i, l in enumerate(model.layers) if l is base_model)
PRE_LAYERS  = model.layers[1:BASE_IDX]        # skip outer InputLayer
POST_LAYERS = model.layers[BASE_IDX+1:]
TAP_MODEL   = tf.keras.Model(
    inputs=base_model.input,
    outputs=[base_model.get_layer(LAST_CONV).output, base_model.output])

def run_gradcam(model, img_path, target_size=(299, 299), alpha=0.45):
    img = tf.keras.utils.load_img(img_path, target_size=target_size)
    raw = tf.keras.utils.img_to_array(img)                 # 0–255 float
    img_array = np.expand_dims(raw, 0)

    probs = model.predict(img_array, verbose=0)[0]
    p_restricted = float(probs[RESTRICTED_IDX].sum())
    label = 'RESTRICTED' if p_restricted > 0.5 else 'UNRESTRICTED'
    conf  = p_restricted if p_restricted > 0.5 else 1 - p_restricted
    top_breed = CLASS_NAMES[int(probs.argmax())]

    with tf.GradientTape() as tape:
        x = tf.cast(img_array, tf.float32)
        for layer in PRE_LAYERS:
            x = layer(x)
        conv_out, base_out = TAP_MODEL(x)
        tape.watch(conv_out)
        z = base_out
        for layer in POST_LAYERS:
            z = layer(z)
        p_restricted_t = tf.reduce_sum(tf.gather(z[0], RESTRICTED_IDX_TF))
    grads  = tape.gradient(p_restricted_t, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))

    heatmap = tf.reduce_sum(conv_out[0] * pooled, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    heatmap = heatmap.numpy()

    heatmap_resized = cv2.resize(heatmap, target_size)
    heatmap_colour  = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    original = raw.astype(np.uint8)
    overlay  = cv2.addWeighted(heatmap_colour, alpha, original, 1 - alpha, 0)

    breed_prefix = '_'.join(os.path.basename(img_path).split('_')[:-1])
    cat = 'restricted' if breed_prefix in RESTRICTED_SET else 'unrestricted'
    out_path = str(GRADCAM_DIR / f'{cat}_{breed_prefix}_gradcam.jpg')

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(original / 255.0)
    axes[0].set_title(f'Original\n{label} ({conf:.1%})  top={top_breed}')
    axes[1].imshow(heatmap_colour[..., ::-1] / 255.0)
    axes[1].set_title('Grad-CAM (restricted-class)')
    axes[2].imshow(overlay[..., ::-1] / 255.0)
    axes[2].set_title('Overlay')
    for a in axes: a.axis('off')
    plt.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f'  saved {out_path}')
    return p_restricted, label, conf, top_breed

print('Grad-CAM function ready.')


## 4.3 — Run Grad-CAM on one image per breed


In [ ]:
def pick_one_per_breed(base_dir):
    paths = []
    for breed in sorted(os.listdir(base_dir)):
        bdir = os.path.join(base_dir, breed)
        if not os.path.isdir(bdir): continue
        imgs = glob.glob(os.path.join(bdir, '*.jpg'))
        if imgs:
            paths.append(random.choice(imgs))
    print(f'  Found {len(paths)} breeds in {os.path.basename(base_dir)}')
    return paths

random.seed(58)
all_imgs = pick_one_per_breed(str(DATA / 'test'))
print(f'\nRunning Grad-CAM on {len(all_imgs)} images...\n')

for i, path in enumerate(all_imgs, 1):
    print(f'[{i}/{len(all_imgs)}] {os.path.basename(path)}')
    run_gradcam(model, path)

print(f'\nGrad-CAM complete. Triplets in: {GRADCAM_DIR}')


## 4.4 — Failure analysis (binary view)


In [ ]:
test_ds = tf.keras.utils.image_dataset_from_directory(
    str(DATA / 'test'), image_size=(299, 299), batch_size=32,
    label_mode='int', class_names=CLASS_NAMES, shuffle=False)

file_paths = test_ds.file_paths
y_true_mc  = np.concatenate([y.numpy() for _, y in test_ds])
y_prob_mc  = model.predict(test_ds).reshape(-1, NUM_CLASSES)
y_prob_bin = y_prob_mc[:, RESTRICTED_IDX].sum(axis=1)
y_true_bin = RESTRICTED_MASK[y_true_mc].astype(int)
y_pred_bin = (y_prob_bin > 0.5).astype(int)

wrong_idx = np.where(y_pred_bin != y_true_bin)[0]

if len(wrong_idx) > 0:
    confidence   = np.abs(y_prob_bin[wrong_idx] - 0.5)
    sorted_wrong = wrong_idx[np.argsort(-confidence)]

    n_show = min(8, len(sorted_wrong))
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for i, idx in enumerate(sorted_wrong[:n_show]):
        img = plt.imread(file_paths[idx])
        true_label = 'Restricted' if y_true_bin[idx] == 1 else 'Unrestricted'
        pred_label = 'Restricted' if y_pred_bin[idx] == 1 else 'Unrestricted'
        true_breed = CLASS_NAMES[y_true_mc[idx]]
        pred_breed = CLASS_NAMES[int(y_prob_mc[idx].argmax())]
        axes[i].imshow(img)
        axes[i].set_title(
            f'True: {true_label} ({true_breed})\n'
            f'Pred: {pred_label} ({pred_breed}, p={y_prob_bin[idx]:.2f})',
            fontsize=9, color='red')
        axes[i].axis('off')

    for i in range(n_show, len(axes)): axes[i].set_visible(False)
    plt.suptitle('Most Confident Binary Misclassifications',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(INFER / 'failure_gallery.png'), dpi=300, facecolor='white')
    plt.show()
    print(f'Failure gallery saved.')
else:
    print('No misclassifications on the test set.')

print(f'\nNotebook 4 complete.')
print(f'  Grad-CAM images: {GRADCAM_DIR}')
print(f'  Failure gallery: {INFER / "failure_gallery.png"}')
